In [84]:
import polars as pl

pl.Config.set_tbl_cols(-1)      # Show all columns
pl.Config.set_tbl_rows(20)      # Optional: show more rows
pl.Config.set_tbl_width_chars(800)  # Increase table width

polars.config.Config

In [85]:
import polars as pl

routes = [
    "Q17",
    "Q24",
    "Q27",
    "Q30",
    "Q31",
]

url = (
    "http://parquet.gtfsrt.io/vehicle_positions"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3ZlaGljbGVQb3NpdGlvbnM"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
      .filter(pl.col("route_id").is_in(routes))
      .select([
      "trip_id",
      "route_id",
      "vehicle_id",
      "direction_id",
    #   "start_time",
      "start_date",
    #   "vehicle_label",
      "latitude",
      "longitude",
      "bearing",
      "stop_id",
      "timestamp"
  ])

      .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(132703, 10)
shape: (5, 10)
┌───────────────────────────────┬──────────┬───────────────┬──────────────┬────────────┬───────────┬────────────┬────────────┬─────────┬────────────┐
│ trip_id                       ┆ route_id ┆ vehicle_id    ┆ direction_id ┆ start_date ┆ latitude  ┆ longitude  ┆ bearing    ┆ stop_id ┆ timestamp  │
│ ---                           ┆ ---      ┆ ---           ┆ ---          ┆ ---        ┆ ---       ┆ ---        ┆ ---        ┆ ---     ┆ ---        │
│ str                           ┆ str      ┆ str           ┆ u32          ┆ str        ┆ f32       ┆ f32        ┆ f32        ┆ str     ┆ u64        │
╞═══════════════════════════════╪══════════╪═══════════════╪══════════════╪════════════╪═══════════╪════════════╪════════════╪═════════╪════════════╡
│ JA_C6-Weekday-118200_Q17_314  ┆ Q17      ┆ MTA NYCT_8440 ┆ 1            ┆ 20260630   ┆ 40.738144 ┆ -73.805496 ┆ 357.220825 ┆ 501388  ┆ 1782863972 │
│ JA_C6-Weekday-118400_MISC_850 ┆ Q30      ┆ MTA NYCT_8434 ┆ 1          

In [86]:
df.null_count()


trip_id,route_id,vehicle_id,direction_id,start_date,latitude,longitude,bearing,stop_id,timestamp
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [87]:
df.write_parquet(
    "raw/vehicle_positions/2026-07-01.parquet"
)

In [88]:
import polars as pl

routes = [
    "Q17",
    "Q24",
    "Q27",
    "Q30",
    "Q31",
]

url = (
    "http://parquet.gtfsrt.io/trip_updates"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3RyaXBVcGRhdGVz"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
      .filter(pl.col("route_id").is_in(routes))
      .select([
        "feed_timestamp",
        "fetch_timestamp",
        "trip_id",
        "route_id",
        "direction_id",
        "start_time",
        "start_date",
        "schedule_relationship",
        "vehicle_id",
        "trip_timestamp",
        "stop_sequence",
        "stop_id",
        "arrival_time",
        "departure_time"
  ])

      .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(3851145, 14)
shape: (5, 14)
┌────────────────┬────────────────────────────────┬──────────────────────────────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┬───────────────┬────────────────┬───────────────┬─────────┬──────────────┬────────────────┐
│ feed_timestamp ┆ fetch_timestamp                ┆ trip_id                      ┆ route_id ┆ direction_id ┆ start_time ┆ start_date ┆ schedule_relationship ┆ vehicle_id    ┆ trip_timestamp ┆ stop_sequence ┆ stop_id ┆ arrival_time ┆ departure_time │
│ ---            ┆ ---                            ┆ ---                          ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---                   ┆ ---           ┆ ---            ┆ ---           ┆ ---     ┆ ---          ┆ ---            │
│ u64            ┆ datetime[μs, UTC]              ┆ str                          ┆ str      ┆ u32          ┆ str        ┆ str        ┆ i32                   ┆ str           ┆ u64            ┆ u32           ┆ str     ┆ i64

In [89]:
df.write_parquet(
    "raw/trip_updates/2026-07-01.parquet"
)

In [90]:
import polars as pl

url = (
    "http://parquet.gtfsrt.io/service_alerts"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL2FsZXJ0cw"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
    .select([
        "feed_timestamp",
        "fetch_timestamp",
        "entity_id",
        "active_period_start",
        "active_period_end",
        "header_text",
        "description_text",
        "url",
        "route_id",
        "route_type",
        "stop_id",
        "trip_id",
        "trip_route_id",
        "trip_direction_id"
    ])
    .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(208877, 14)
shape: (5, 14)
┌────────────────┬────────────────────────────────┬─────────────────────────────────┬─────────────────────┬───────────────────┬─────────────────────────────────┬─────────────────────────────────┬──────┬──────────┬────────────┬─────────┬─────────┬───────────────┬───────────────────┐
│ feed_timestamp ┆ fetch_timestamp                ┆ entity_id                       ┆ active_period_start ┆ active_period_end ┆ header_text                     ┆ description_text                ┆ url  ┆ route_id ┆ route_type ┆ stop_id ┆ trip_id ┆ trip_route_id ┆ trip_direction_id │
│ ---            ┆ ---                            ┆ ---                             ┆ ---                 ┆ ---               ┆ ---                             ┆ ---                             ┆ ---  ┆ ---      ┆ ---        ┆ ---     ┆ ---     ┆ ---           ┆ ---               │
│ u64            ┆ datetime[μs, UTC]              ┆ str                             ┆ u64                 ┆ u64            

In [91]:
df.null_count()


feed_timestamp,fetch_timestamp,entity_id,active_period_start,active_period_end,header_text,description_text,url,route_id,route_type,stop_id,trip_id,trip_route_id,trip_direction_id
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,25622,0,0,208877,152314,208877,208877,56563,56563,56563


In [92]:
df.write_parquet(
    "raw/service_alerts/2026-07-01.parquet"
)